---
title: "Publish: Gather the Aggregated Data and Publish to DataVerse"
engine: jupyter
---

## publish 

> This is the `publish` module for the ERA5 dataset pipeline. It defines a functions that make use of the `pyDataverse` library and API to publish our outputs to the Harvard Dataverse.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

First, we'll test out the API by pinging the Harvard DataVerse

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
import hydra
import yaml
import json
from tqdm import tqdm
from pyprojroot import here

In [ ]:
api_token_file = here() / "sandbox/dataverse_api_key.yml"
with open(api_token_file, "r") as f:
    config = yaml.load(f, Loader=yaml.BaseLoader)

Now, following the [docs]() for the dataverse tutorial, load a NativeAPI up:

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
from pyDataverse.api import NativeApi

The NativeAPI is a catchall API object to be able to do general stuff:

In [ ]:
api = NativeApi(config['base_url'], config['api_token'])
resp=api.get_info_version()
#resp.text()

In [ ]:
resp.json()

Looks good! Now that we know that it works, we can think more
about how to publish data there.

## Harvard Dataverse

Let's create a dummy dataset with the components we're
planning to upload, and then upload and promptly delete it.

To do that, we must import the `models` module and create a Dataset object:

In [ ]:
from pyDataverse.models import Dataset

In [ ]:
ds = Dataset()

This `ds` object is pretty straightforward since it doesn't contain anything yet:

In [ ]:
ds.get()

We can populate the object from the dummy data on the github repo:

In [ ]:
from pyDataverse.utils import read_file
from urllib.request import urlretrieve
import tempfile

In [ ]:
# url for dummy data
url = "https://raw.githubusercontent.com/gdcc/pyDataverse/refs/heads/main/tests/data/user-guide/dataset.json"


with tempfile.NamedTemporaryFile(mode='w+') as tmp:
    urlretrieve(url, tmp.name)
    ds.from_json(read_file(tmp.name))

We have to validate the JSON correctly:

In [ ]:
ds.validate_json()

Modifying it is easy:

In [ ]:
ds.set({"title": "Youth from Austria 2005"})
ds.get()

Now, to create the dataset we use the API:

In [ ]:
#| eval: false
# this is only run in interactive sessions for demo purposes
resp = api.create_dataset(":root", ds.json())

If you caught the `resp` object, it contains the PID for the newly created dataset.

However, if you didn't you can use the SearchAPI to find it:

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
from pyDataverse.api import SearchApi

In [ ]:
search_api = SearchApi(config['base_url'], config['api_token'])

In [ ]:
#| eval: false
#

resp = search_api.search("Youth from Austria", data_type="dataset")
results = resp.json()['data']['items']
result = [x for x in results if "Youth from Austria" in x['name']][0]
result

In [ ]:
#| eval: false
pid = result['global_id']

Now to look at the data we created using the NativeAPI again, and delete the dataset:

In [ ]:
#| eval: false
uploaded_ds = api.get_dataset(pid)
uploaded_ds.json()['data']

resp = api.delete_dataset(pid)
resp.json()

With that understanding, we can develop a quick module to do the following:

1. Make the dataset LEGO Compatible
2. Upload and publish the data to dataverse

## LEGO Compatibility

Let's take an example file to use as a model for LEGO compatibility

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
import geopandas as gpd
import pandas as pd
import re
import glob

In [ ]:
ex = gpd.read_parquet(here() / "bld/2009_06_madagascar_day_swvl1_mean.parquet")
ex.describe()

We know that the LEGO data model should look like this:

```
<main lab folder>/lego
├── <domain>
│   ├── <subdomain>__<data_source>
│   │   ├── <geo_resolution>__<time_resolution>
│   │   │   ├── <filename>_yyyy.parquet
```

So, for the above file, we'll end up with the LEGO path `data/environmental/exposures_era5/healthshed_monthly/dewpoint_2024.parquet`. In it, we should have the following columns:


```
healthshed_id  year month day stat_1 stat_2 ... stat_n   
```


This means we should read in all of the exposures for a single timepoint at once. 
I think the smart thing to do is use a glob string to gather all of the pertinent files.
This will be the first function we export to the library:

In [0]:
#| echo: false
#| output: asis
show_doc(gather_exposure_geodataframes)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/publish.py#L32){target="_blank" style="float:right; font-size:smaller"}

### gather_exposure_geodataframes

>      gather_exposure_geodataframes (glob_string:str, polygon_id:str,
>                                     exposure:str)

*Read in a list of geo dataframes from the same time frame and merge them*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| glob_string | str | string for the path to search for the pertinent files |
| polygon_id | str | the string signifying the healthshed ID of the polygon |
| exposure | str | the exposure name |
| **Returns** | **list** |  |

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: # 

def gather_exposure_geodataframes(
    glob_string: str,   # string for the path to search for the pertinent files
    polygon_id: str,    # the string signifying the healthshed ID of the polygon
    exposure: str       # the exposure name
    )-> list:
    "Read in a list of geo dataframes from the same time frame and merge them"

    # first get the initial one so we have the polygon ID and geometry
    frames = glob.glob(str(glob_string))
    initial_gdf=gpd.read_parquet(frames[0])
    merged_df = []
  
    for f in tqdm(frames, desc="Processing files"):
        # read in as a regular dataframe by ignoring geometry
        df = gpd.read_parquet(f).drop(["geometry"], axis=1) 
        
        # get the year and month
        # Extract year and month
        search_str = rf'_{exposure}_(\d{{4}})_(\d{{1,2}})\.parquet$'
        match = re.search(search_str, f)

        if match:
            year = int(match.group(1))
            month = int(match.group(2))
            #print(f"Year: {year}, Month: {month}")
        else:
            raise ValueError(f"Could not extract year and month from filename: {search_str} {f}")
            
        df['exposure'] = exposure
        df['month'] = month
        df['year'] = year

        # Step 1: Melt all day columns (leave 'month' and 'year' as identifiers)
        df_long = df.melt(id_vars=[polygon_id, "exposure", "year", "month"], var_name="day_stat", value_name="value")

        # Step 2: Extract day and stat type from column names
        # Example column: "day_01_daily_mean"
        df_long[["day", "stat"]] = df_long["day_stat"].str.extract(r"day_(\d{2})_daily_(mean|max|min|total)")

        # Optional: convert 'day' and month to integer
        df_long["day"] = df_long["day"].astype(int)
        df_long["month"] = df_long["month"].astype(int)

        # Drop the original combined column
        df_long = df_long.drop(columns="day_stat")

        # Reorder columns
        df_long = df_long[[polygon_id, "exposure", "year", "month", "day", "stat", "value"]]

        df_long = df_long.sort_values(by=["year", "month", "day"])
        df_clean = df_long.pivot(index=[polygon_id, "exposure", "year", "month", "day"], columns="stat", values="value").reset_index()
        merged_df.append(df_clean)

    return [pd.concat(merged_df).reset_index(drop=True), initial_gdf[[polygon_id, "geometry"]]]

In [ ]:
frames = here() / "data" / "testing" / "*madagascar*"

merged = gather_exposure_geodataframes(frames, "fs_uid", "2m_dewpoint_temperature")
merged[0].describe()

This returns one file with all of the geometries and one file
with the statistics and exposures.

Now, with this, we can move on. The dataset was created in the UI and is available via search and test out how to upload it:

In [ ]:
resp = search_api.search("ERA5", data_type="dataset")

results = resp.json()['data']['items']

result = [x for x in results if "ERA5" in x['name']][0]
era5_pid = result['global_id']
result

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #

from pyDataverse.models import Datafile
import os
import pathlib

We'll upload directly from file. In the case of ERA5 vs. LEGO, we
store the file on disk as LEGO hierarchy, but to upload it to dataverse
using a flat filename (since creating subdatasets to represent directories might be 
a bit of a hassle)

In [ ]:
# assuming the file has a path on disk like:
f_out = "environmental/exposures_era5/healthshed_daily/dewpoint_2024.parquet"
os.makedirs(here() / "data" / "testing" / os.path.dirname(f_out), exist_ok=True)
aggregations, geo = merged
aggregations.to_parquet(here() / "data" / "testing" / f_out, index=False)

datafile = Datafile()
datafile.set({
    # the id of the era5 dataset 
    "pid": era5_pid,
    # the path to the file on disk goes here
    "filename": str(here() / "data" / "testing" / f_out),
    # use the "label" to name the file
    "label": f_out.replace("/", "-")
})

In [ ]:
#| eval: false
resp = api.upload_datafile(era5_pid, str(here() / "data" / "testing" / f_out), datafile.json())

Pretty simple!

Now, we just need a main function to upload this data. The final upload is one file per
exposure per year, so these should be the variables we gather data for.

We should get some functionality to gather the groups of these files automatically, based on
the hydra config:

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
from hydra import initialize, compose
from omegaconf import OmegaConf, DictConfig
from tqdm import tqdm

In [ ]:
target_dir = here() / "data" / "intermediate"

try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

cfg.development_mode = False
#cfg.query['year'] = 2017
#cfg.query['month'] = 11
#cfg.query['geography'] = "nepal"

In [0]:
#| echo: false
#| output: asis
show_doc(main)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L302){target="_blank" style="float:right; font-size:smaller"}

### main

>      main (cfg:omegaconf.dictconfig.DictConfig)

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #

@hydra.main(version_base=None, config_path="../../conf", config_name="config")
def main(cfg: DictConfig) -> None:

    variables_dict = {
        "2m_temperature": "t2m",
        "2m_dewpoint_temperature": "d2m",
        "volumetric_soil_water_layer_1": "swvl1",
        "total_precipitation": "tp"
    }

    print(OmegaConf.to_yaml(cfg))

    #prep dataverse
    api_token_file = here() / "sandbox/dataverse_api_key.yml"
    with open(api_token_file, "r") as f:
        apiconfig = yaml.load(f, Loader=yaml.BaseLoader)
    api = NativeApi(apiconfig['base_url'], apiconfig['api_token'])
    search_api = SearchApi(apiconfig['base_url'], apiconfig['api_token'])
    resp = search_api.search("ERA5", data_type="dataset")

    results = resp.json()['data']['items']

    result = [x for x in results if "ERA5" in x['name']][0]
    era5_pid = result['global_id']

    for geography in cfg.geographies:
        for year in cfg.query['year']:
            for variable, v in variables_dict.items():
                
                print(f"Processing {geography} for {variable} in {year}")
                glob_string = here() / "data" / "intermediate" / f"*{geography}*{variable}*{year}*"
                print(f"Glob: {glob_string}")
                polygon_id = cfg.geographies[geography]['unique_id']
                print(f"polygon_id: {polygon_id}")
                merged = gather_exposure_geodataframes(glob_string, polygon_id, variable)
                print(merged[0].head())
                print(merged[1].head())

                output_dir = here() / "data" / "output" 
                
                f_out = f"environmental/exposures_era5/healthshed_daily/{geography}_{v}_{year}.parquet"
                os.makedirs(output_dir / os.path.dirname(f_out), exist_ok=True)
                output_path = output_dir / f_out

                print(f"Writing to {output_path}")
                merged[0].to_parquet(output_path, index=False)
                

                print(f"Uploading {f_out.replace('/', '-')} to Dataverse...")
                # upload to dataverse
                datafile = Datafile()
                datafile.set({
                    "pid": era5_pid,
                    "filename": str(output_path),
                    "label": f_out.replace("/", "-")
                })

                resp = api.upload_datafile(era5_pid, output_path, datafile.json())
                assert resp.json()['status'] == "OK", f"Failed to upload datafile: {resp.text}"
        
        # also save the geometry for the region 
        merged[1].to_parquet(output_path.parent / f"{geography}_geometry.parquet", index=False)

        # and upload it to dataverse
        datafile = Datafile()
        datafile.set({
            "pid": era5_pid,
            "filename": str(output_path.parent / f"{geography}_geometry.parquet"),
            "label": f"{geography}_geometry.parquet"
        })

        resp = api.upload_datafile(era5_pid, output_path.parent / f"{geography}_geometry.parquet", datafile.json())
        assert resp.json()['status'] == "OK", f"Failed to upload geometry datafile: {resp.text}"

    print("All files processed and uploaded successfully.")